In [1]:
import os
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
from cmapPy.pandasGEXpress.parse import parse
import pickle
from tqdm import tqdm
from pathlib import Path

In [2]:

# print current directory
print("Current working directory:", os.getcwd())
DATA_DIR = "data_phase1"
GCTX_FILE = os.path.join(DATA_DIR, "level5.gctx")
PERTURB_FILE = os.path.join(DATA_DIR, "perturbation_pairs.csv")
OUTPUT_DIR = "experiment_data/raw"

Current working directory: /sc/home/johanna.dahlkemper/source_detection_in_grns/pathway_experiment


In [3]:
# Load perturbation info
df_perturbation_pairs = pd.read_csv(PERTURB_FILE, sep="\t")
df_perturbation_pairs

,unperturbed_sig_id,perturbed_sig_id,perturbed_gene,perturbed_gene_id,cell_line,perturbation_type
0,CNS001_A375_96H:UNTRT:-666,CGS001_A375_96H:A2M:1,A2M,2,A375,trt_sh.cgs
1,CNS001_A375_96H:UNTRT:-666,CGS001_A375_96H:ABCA1:1,ABCA1,19,A375,trt_sh.cgs
2,CNS001_A375_96H:UNTRT:-666,CGS001_A375_96H:ABCB1:1,ABCB1,5243,A375,trt_sh.cgs
3,CNS001_A375_96H:UNTRT:-666,CGS001_A375_96H:ABCF2:1,ABCF2,10061,A375,trt_sh.cgs
4,CNS001_A375_96H:UNTRT:-666,CGS001_A375_96H:ABL1:1,ABL1,25,A375,trt_sh.cgs
...,...,...,...,...,...,...
76674,TAK004_U2OS_96H:CMAP-000:1,TAK004_U2OS_96H:TRCN0000370007:1,WWTR1,25937,U2OS,trt_sh
76675,TAK004_U2OS_96H:CMAP-000:1,TAK004_U2OS_96H:TRCN0000370678:1,GRB10,2887,U2OS,trt_sh
76676,TAK004_U2OS_96H:CMAP-000:1,TAK004_U2OS_96H:TRCN0000370697:1,GRB10,2887,U2OS,trt_sh
76677,TAK004_U2OS_96H:CMAP-000:1,TAK004_U2OS_96H:TRCN0000370751:1,GRB10,2887,U2OS,trt_sh


In [4]:
with open(f"experiment_data/gene_to_idx.pkl", "rb") as f:
    L1000_to_idx_map = pickle.load(f)
with open(f"experiment_data/kegg_to_L1000_map.pkl", "rb") as f:
    kegg_to_L1000_map = pickle.load(f)

L1000_to_kegg_map = {v: k for k, v in kegg_to_L1000_map.items()}

In [5]:
genes_of_interest = [str(L1000_to_kegg_map[gene]) for gene in L1000_to_idx_map.keys()]
sigs_of_interest = df_perturbation_pairs['perturbed_sig_id'].tolist() + df_perturbation_pairs['unperturbed_sig_id'].tolist()
sigs_of_interest = list(set(sigs_of_interest))
print(f"Number of unique genes of interest: {len(genes_of_interest)}")
print(f"Number of unique sigs of interest: {len(sigs_of_interest)}")

Number of unique genes of interest: 2655
Number of unique sigs of interest: 76704


In [6]:
gctx_meta = parse(GCTX_FILE)
gctx_meta

/sc/home/johanna.dahlkemper/source_detection_in_grns/.venv/lib/python3.12/site-packages/cmapPy/pandasGEXpress/parse_gctx.py:275: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  meta_df = meta_df.apply(lambda x: pd.to_numeric(x, errors="ignore"))
/sc/home/johanna.dahlkemper/source_detection_in_grns/.venv/lib/python3.12/site-packages/cmapPy/pandasGEXpress/parse_gctx.py:275: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  meta_df = meta_df.apply(lambda x: pd.to_numeric(x, errors="ignore"))


In [7]:
data_gct = parse(GCTX_FILE, rid=genes_of_interest, cid=sigs_of_interest)
data_gct.data_df.shape

/sc/home/johanna.dahlkemper/source_detection_in_grns/.venv/lib/python3.12/site-packages/cmapPy/pandasGEXpress/parse_gctx.py:275: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  meta_df = meta_df.apply(lambda x: pd.to_numeric(x, errors="ignore"))
/sc/home/johanna.dahlkemper/source_detection_in_grns/.venv/lib/python3.12/site-packages/cmapPy/pandasGEXpress/parse_gctx.py:275: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  meta_df = meta_df.apply(lambda x: pd.to_numeric(x, errors="ignore"))


(2655, 76704)

In [8]:
kegg_to_L1000_idx_map = {}
for kegg_id, l1000_gene in kegg_to_L1000_map.items():
    if l1000_gene in L1000_to_idx_map:
        idx = L1000_to_idx_map[l1000_gene]
        kegg_to_L1000_idx_map[kegg_id] = idx
sorted_kegg_ids = sorted(kegg_to_L1000_idx_map.keys(), key=lambda kegg_id: kegg_to_L1000_idx_map[kegg_id])

In [9]:
# pick some KEGG IDs to test
test_ids = sorted_kegg_ids[:5]

# cache first-column values before sorting
colname = data_gct.data_df.columns[0]
before_vals = {k: data_gct.data_df.loc[k, colname] for k in test_ids}

# perform sorting
data_gct.data_df = data_gct.data_df.loc[sorted_kegg_ids]

# verify that the order is correct
for i, kegg_id in enumerate(data_gct.data_df.index):
    l1000_gene = kegg_to_L1000_map.get(kegg_id, None)
    if l1000_gene is not None:
        expected_idx = L1000_to_idx_map[l1000_gene]
        if expected_idx != i:
            print(f"Mismatch at position {i}: kegg_id {kegg_id} maps to {l1000_gene} which has index {expected_idx}")

print("First 5 L1000 indices:", list(L1000_to_idx_map.keys())[:5])
print("Kegg IDs for first 5 L1000 indices:", [L1000_to_kegg_map[idx] for idx in list(L1000_to_idx_map.keys())[:5]])
print("First 5 sorted KEGG IDs:", sorted_kegg_ids[:5])
print("First 5 row indices in data_gct.data_df:", data_gct.data_df.index[:5].tolist())

# test whether values moved with their index
for k in test_ids:
    new_pos = data_gct.data_df.index.get_loc(k)
    after_val = data_gct.data_df.iloc[new_pos][colname]

    if after_val != before_vals[k]:
        print(f"Value mismatch for {k}: expected {before_vals[k]}, found {after_val} at new position {new_pos}")
    else:
        print(f"OK: {k} kept its value ({after_val}) at new position {new_pos}")


First 5 L1000 indices: ['CYBB', 'NOX1', 'NOX4', 'AGER', 'NFKB1']
Kegg IDs for first 5 L1000 indices: ['1536', '27035', '50507', '177', '4790']
First 5 sorted KEGG IDs: ['1536', '27035', '50507', '177', '4790']
First 5 row indices in data_gct.data_df: ['1536', '27035', '50507', '177', '4790']
OK: 1536 kept its value (1.5106550455093384) at new position 0
OK: 27035 kept its value (0.5104408264160156) at new position 1
OK: 50507 kept its value (-1.3069193363189697) at new position 2
OK: 177 kept its value (1.3157113790512085) at new position 3
OK: 4790 kept its value (-1.3790489435195923) at new position 4


In [10]:
num_possible_sources=df_perturbation_pairs['perturbed_gene_id'].nunique()
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Create a mapping from (gene_id, cell_line, pert_type) -> group_id ONCE (outside loop)
print("Creating group mapping...")
group_mapping = {}
group_ids_list = []
unique_groups = []

for idx, row in df_perturbation_pairs.iterrows():
    # Create group key
    group_key = (
        row['perturbed_gene_id'],
        row.get('cell_line', 'unknown'),
        row.get('perturbation_type', 'unknown')
    )

    # Assign group ID (incrementing)
    if group_key not in group_mapping:
        group_id = len(group_mapping)
        group_mapping[group_key] = group_id
        unique_groups.append(group_key)
    else:
        group_id = group_mapping[group_key]
    
    group_ids_list.append(group_id)

print(f"Created {len(unique_groups)} unique groups")

# Convert to numpy array for storage
group_ids = np.array(group_ids_list, dtype=np.int32)

# Now loop through data creation and save files
print("Creating data files...")
for idx, row in tqdm(df_perturbation_pairs.iterrows(), total=df_perturbation_pairs.shape[0]):
    unpert_sig_id = row['unperturbed_sig_id']
    pert_sig_id = row['perturbed_sig_id']
    pert_gene_id = row['perturbed_gene_id']
    L1000_gene = kegg_to_L1000_map.get(str(pert_gene_id), None)

    # Extract gene expression vectors
    original = data_gct.data_df[unpert_sig_id].values.astype(np.float32)
    perturbed = data_gct.data_df[pert_sig_id].values.astype(np.float32)
    difference = perturbed - original

    # One-hot perturbation indicator
    binary_perturbation_indicator = np.zeros(len(genes_of_interest), dtype=np.float32)
    binary_perturbation_indicator[L1000_to_idx_map[L1000_gene]] = 1.0

    # Create Data object
    data_obj = Data(
        original=original,
        perturbed=perturbed,
        difference=difference,
        binary_perturbation_indicator=binary_perturbation_indicator,
        perturbed_gene=L1000_gene,
        gene_mapping=L1000_to_idx_map,
        num_nodes=len(L1000_to_idx_map),
        num_possible_sources=num_possible_sources
    )

    # Save individual file
    out_file = os.path.join(OUTPUT_DIR, f"{idx}.pt")
    torch.save(data_obj, out_file)

# Save metadata ONCE after all data files to the shared data path
print("Saving metadata...")
sample_metadata = {
    "group_ids": group_ids,
    "unique_groups": unique_groups,
    "group_mapping": group_mapping,
    "n_samples": len(group_ids),
    "n_unique_groups": len(unique_groups),
}

# Determine the shared data path (same directory where raw data is stored)
SHARED_DATA_PATH = Path(OUTPUT_DIR).parent
SHARED_DATA_PATH.mkdir(parents=True, exist_ok=True)

metadata_file = SHARED_DATA_PATH / "sample_metadata.pt"
torch.save(sample_metadata, metadata_file)

print(f"Done! All data objects saved.")
print(f"Metadata saved to {metadata_file}: {len(group_ids)} samples in {len(unique_groups)} groups")

Creating group mapping...
Created 28261 unique groups
Creating data files...


100%|██████████| 76679/76679 [10:05<00:00, 126.70it/s]


Saving metadata...
Done! All data objects saved.
Metadata saved to experiment_data/sample_metadata.pt: 76679 samples in 28261 groups
